In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer


c:\Users\chris\Documents\GitHub\ML_DL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

These values keep the models small enough for a lab session.


In [2]:
# Select GPU if available.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define how many sequences are processed in one training step.
batch_size = 32

# Define how many tokens each input sequence contains.
block_size = 128

# Define model and training hyperparameters.
embedding_dim = 128
hidden_dim = 256
num_layers = 2
learning_rate = 1e-3
max_steps = 3000
eval_interval = 300

# Make the experiment more reproducible.
torch.manual_seed(42)

print("Device:", device)


Device: cuda


## 3. Load WikiText-2

WikiText-2 is used as one long text source for next-token prediction.


In [3]:
# Load WikiText-2 from Hugging Face Datasets.
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# Combine all lines into one long training text.
train_text = "\n".join(dataset["train"]["text"])

# Combine all lines into one long validation text.
val_text = "\n".join(dataset["validation"]["text"])

# Print basic dataset information.
print("Train characters:", len(train_text))
print("Validation characters:", len(val_text))
print("\nPreview:")
print(train_text[:500])


Train characters: 10929707
Validation characters: 1145909

Preview:

 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs


## 4. Character-Level Tokenization

A neural network cannot work directly with text.

Therefore, we convert each character into an integer ID.

Here, one token means one character.


In [4]:
# Collect all unique characters that appear in the training text.
chars = sorted(list(set(train_text)))

# Count how many different characters the model can predict.
char_vocab_size = len(chars)

# Map each character to a unique integer ID.
stoi = {ch: i for i, ch in enumerate(chars)}

# Map each integer ID back to its original character.
itos = {i: ch for i, ch in enumerate(chars)}

# Convert text into character IDs.
def encode_char(s):
    return [stoi[c] for c in s if c in stoi]

# Convert character IDs back into readable text.
def decode_char(indices):
    return "".join([itos[i] for i in indices])

# Encode the full training and validation text.
train_char_data = torch.tensor(encode_char(train_text), dtype=torch.long)
val_char_data = torch.tensor(encode_char(val_text), dtype=torch.long)

print("Character vocabulary size:", char_vocab_size)
print("Training character tokens:", len(train_char_data))
print("Validation character tokens:", len(val_char_data))


Character vocabulary size: 1013
Training character tokens: 10929707
Validation character tokens: 1145751


## 5. Batching for Next-Character Prediction

The input sequence `x` is the text the model sees.

The target sequence `y` is shifted by one character.

```text
x: The capita
y: he capital
```

So the model learns to predict the next character at every position.


In [5]:
def get_batch_char(split):
    # Select training or validation data.
    source = train_char_data if split == "train" else val_char_data

    # Choose random start positions.
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))

    # Create input sequences.
    x = torch.stack([source[i:i + block_size] for i in ix])

    # Create target sequences shifted by one character.
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])

    # Move tensors to CPU or GPU.
    return x.to(device), y.to(device)

# Inspect one batch.
xb, yb = get_batch_char("train")
print("x shape:", xb.shape)
print("y shape:", yb.shape)
print("\nExample input:")
print(decode_char(xb[0][:80].tolist()))
print("\nExample target:")
print(decode_char(yb[0][:80].tolist()))


x shape: torch.Size([32, 128])
y shape: torch.Size([32, 128])

Example input:
( RIAJ ) for shipments of 250 @,@ 000 copies . In Australia , The Remix entered 

Example target:
 RIAJ ) for shipments of 250 @,@ 000 copies . In Australia , The Remix entered t


## 6. LSTM Language Model

The LSTM reads the sequence from left to right and predicts the next character.

The loss is cross-entropy, because each position is a classification task over the vocabulary.


In [6]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()

        # Convert character IDs into dense vectors.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Process the sequence from left to right.
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # Predict one vocabulary logit vector per time step.
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, idx, targets=None):
        # Convert token IDs into embeddings.
        embeddings = self.embedding(idx)

        # Process the sequence with the LSTM.
        lstm_out, _ = self.lstm(embeddings)

        # Convert hidden states into next-character logits.
        logits = self.fc(lstm_out)

        # Compute loss only during training.
        loss = None
        if targets is not None:
            B, T, C = logits.shape

            # Flatten batch and time dimensions for cross-entropy.
            logits_flat = logits.reshape(B * T, C)

            # Flatten targets to match the logits.
            targets_flat = targets.reshape(B * T)

            # Compare predicted next-character logits with true next characters.
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss


In [7]:
# Create the LSTM model.
lstm_model = LSTMLanguageModel(
    vocab_size=char_vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers
).to(device)

# Create the optimizer.
lstm_optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=learning_rate)

print("LSTM parameters:", sum(p.numel() for p in lstm_model.parameters()))


LSTM parameters: 1311605


### Train the LSTM



In [8]:
@torch.no_grad()
def estimate_lstm_loss():
    # Switch to evaluation mode.
    lstm_model.eval()

    # Store average losses for train and validation.
    results = {}

    for split in ["train", "val"]:
        losses = []

        # Average multiple batches for a more stable estimate.
        for _ in range(20):
            xb, yb = get_batch_char(split)
            _, loss = lstm_model(xb, yb)
            losses.append(loss.item())

        results[split] = sum(losses) / len(losses)

    # Switch back to training mode.
    lstm_model.train()

    return results


In [9]:
# Train the LSTM model.
lstm_model.train()

for step in range(max_steps):
    # Get one training batch.
    xb, yb = get_batch_char("train")

    # Compute predictions and loss.
    _, loss = lstm_model(xb, yb)

    # Reset gradients.
    lstm_optimizer.zero_grad()

    # Compute gradients.
    loss.backward()

    # Update model parameters.
    lstm_optimizer.step()

    # Print progress.
    if step % eval_interval == 0:
        losses = estimate_lstm_loss()
        print(f"Step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


Step 0: train loss 6.8703, val loss 6.8703
Step 300: train loss 2.6012, val loss 2.5992
Step 600: train loss 2.2761, val loss 2.2753
Step 900: train loss 2.0973, val loss 2.0990
Step 1200: train loss 2.0041, val loss 1.9889
Step 1500: train loss 1.9124, val loss 1.9023
Step 1800: train loss 1.8443, val loss 1.8443
Step 2100: train loss 1.8229, val loss 1.8010
Step 2400: train loss 1.7657, val loss 1.7652
Step 2700: train loss 1.7343, val loss 1.7348


### Generate Text with the LSTM

Generation is autoregressive: predict one character, append it, and repeat.


In [ ]:
@torch.no_grad()
def generate_lstm(model, start_text, max_new_chars=300, temperature=1.0):
    # Switch to evaluation mode.
    model.eval()

    # Convert the prompt into character IDs.
    context = torch.tensor([encode_char(start_text)], dtype=torch.long).to(device)

    for _ in range(max_new_chars):
        # Predict logits for the current context.
        logits, _ = model(context)

        # Use only the last time step for generation.
        logits = logits[:, -1, :]

        # Lower temperature is safer; higher temperature is more random.
        logits = logits / temperature

        # Convert logits into probabilities.
        probs = F.softmax(logits, dim=-1)

        # Sample the next character.
        next_char = torch.multinomial(probs, num_samples=1)

        # Append the new character to the context.
        context = torch.cat([context, next_char], dim=1)

    # Convert generated IDs back into text.
    return decode_char(context[0].tolist())

print(generate_lstm(
    lstm_model,
    start_text="The capital of France is",
    max_new_chars=15,
    temperature=0.75
))


The capital of France is are explook or


In [ ]:

print(generate_lstm(
    lstm_model,
    start_text="The capital of France is",
    max_new_chars=15,
    temperature=0.75
))


## Try it with different hyperparameter



In [11]:
temperature = 0.5
temperature = 1.0
temperature = 1.5

block_size = 64
block_size = 128
block_size = 256

hidden_dim = 128
hidden_dim = 256
hidden_dim = 512

## 7. Character-Level GPT-Style Transformer

This model uses the Hugging Face GPT-2 architecture, but it is **not pretrained**.

The weights start randomly.

The tokenizer is still our own character-level tokenizer.


In [12]:
# Define a small GPT-2 style configuration for character tokens.
char_config = GPT2Config(
    vocab_size=char_vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=128,
    n_layer=4,
    n_head=4,
    bos_token_id=0,
    eos_token_id=0,
    pad_token_id=0,
)

# Create an untrained GPT-2 style model with random weights.
char_gpt = GPT2LMHeadModel(char_config).to(device)

print("Character GPT parameters:", char_gpt.num_parameters())


Character GPT parameters: 955776


### Important Detail: Labels in Hugging Face GPT Models

For `GPT2LMHeadModel`, pass `labels=input_ids`.

Hugging Face shifts the labels internally for next-token prediction.

So we do **not** pass the manually shifted `y` as labels here.


In [13]:
# Create a separate optimizer for the character-level GPT.
char_gpt_optimizer = torch.optim.AdamW(char_gpt.parameters(), lr=1e-3)

# Define number of training steps.
char_gpt_steps = 3000

# Train the untrained character-level GPT.
char_gpt.train()

for step in range(char_gpt_steps):
    # Get one character batch.
    xb, _ = get_batch_char("train")

    # Pass labels=xb because Hugging Face shifts labels internally.
    outputs = char_gpt(input_ids=xb, labels=xb)

    # Extract the cross-entropy loss.
    loss = outputs.loss

    # Reset gradients.
    char_gpt_optimizer.zero_grad()

    # Compute gradients.
    loss.backward()

    # Update model parameters.
    char_gpt_optimizer.step()

    # Print progress.
    if step % eval_interval == 0:
        print(f"Step {step}, loss: {loss.item():.4f}")


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step 0, loss: 6.9464


[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (0) is identical to the `bos_token_id` (0), `eos_token_id` (0), or the `sep_token_id` (None), and your input is not padded.


Step 300, loss: 2.4855
Step 600, loss: 2.2939
Step 900, loss: 2.1467
Step 1200, loss: 1.9822
Step 1500, loss: 1.8436
Step 1800, loss: 1.7503
Step 2100, loss: 1.7475
Step 2400, loss: 1.6903
Step 2700, loss: 1.6378


In [14]:
# Define a starting text.
start_text = "The capital of France is"

# Convert starting text into character IDs.
input_ids = torch.tensor([encode_char(start_text)], dtype=torch.long).to(device)

# Generate new character IDs.
output_ids = char_gpt.generate(
    input_ids,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_k=20,
    pad_token_id=0,
)

# Convert character IDs back into readable text.
print(decode_char(output_ids[0].tolist()))


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The capital of France issued this song for the is operas the stilll protestial on Marisan 2010 , 200 , w


## 8. GPT-2 Tokenizer

Now we use the GPT-2 tokenizer.

This tokenizer uses subword tokens instead of single characters.

A token can be a word, part of a word, punctuation, or a space-aware word part.


In [15]:
# Load the GPT-2 tokenizer for subword tokenization.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Use the end-of-text token as padding token.
tokenizer.pad_token = tokenizer.eos_token

# Convert training text into GPT-2 token IDs.
train_gpt2_ids = tokenizer.encode(train_text, add_special_tokens=False)

# Convert validation text into GPT-2 token IDs.
val_gpt2_ids = tokenizer.encode(val_text, add_special_tokens=False)

# Store tokenized data as tensors.
train_gpt2_data = torch.tensor(train_gpt2_ids, dtype=torch.long)
val_gpt2_data = torch.tensor(val_gpt2_ids, dtype=torch.long)

print("GPT-2 vocabulary size:", tokenizer.vocab_size)
print("Training GPT-2 tokens:", len(train_gpt2_data))
print("Validation GPT-2 tokens:", len(val_gpt2_data))


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2403644 > 1024). Running this sequence through the model will result in indexing errors


GPT-2 vocabulary size: 50257
Training GPT-2 tokens: 2403644
Validation GPT-2 tokens: 248461


### Tokenizer Comparison on the Dataset

Both tokenizers process the same dataset.

The character tokenizer creates many short tokens.

The GPT-2 tokenizer creates fewer but larger subword tokens.


In [16]:
# Encode the full training text with the character-level tokenizer.
char_ids_full = encode_char(train_text)

# Encode the full training text with the GPT-2 tokenizer.
gpt2_ids_full = train_gpt2_ids

# Convert the first GPT-2 token IDs into readable token strings.
gpt2_tokens_preview = tokenizer.convert_ids_to_tokens(gpt2_ids_full[:50])

print("Tokenizer comparison on WikiText-2 train split")
print("-" * 62)
print(f"{'Metric':30s} {'Character-Level':>15s} {'GPT-2':>15s}")
print("-" * 62)
print(f"{'Vocabulary size':30s} {len(chars):15d} {tokenizer.vocab_size:15d}")
print(f"{'Number of tokens':30s} {len(char_ids_full):15d} {len(gpt2_ids_full):15d}")
print(f"{'Characters per token':30s} {len(train_text)/len(char_ids_full):15.2f} {len(train_text)/len(gpt2_ids_full):15.2f}")

print("\nFirst 50 character-level tokens:")
print(list(train_text[:50]))

print("\nFirst 50 GPT-2 tokens:")
print(gpt2_tokens_preview)


Tokenizer comparison on WikiText-2 train split
--------------------------------------------------------------
Metric                         Character-Level           GPT-2
--------------------------------------------------------------
Vocabulary size                           1013           50257
Number of tokens                      10929707         2403644
Characters per token                      1.00            4.55

First 50 character-level tokens:
['\n', ' ', '=', ' ', 'V', 'a', 'l', 'k', 'y', 'r', 'i', 'a', ' ', 'C', 'h', 'r', 'o', 'n', 'i', 'c', 'l', 'e', 's', ' ', 'I', 'I', 'I', ' ', '=', ' ', '\n', '\n', '\n', ' ', 'S', 'e', 'n', 'j', 'ō', ' ', 'n', 'o', ' ', 'V', 'a', 'l', 'k', 'y', 'r', 'i']

First 50 GPT-2 tokens:
['Ċ', 'Ġ=', 'ĠV', 'alky', 'ria', 'ĠChronicles', 'ĠIII', 'Ġ=', 'Ġ', 'ĊĊ', 'Ċ', 'ĠSen', 'j', 'Åį', 'Ġno', 'ĠV', 'alky', 'ria', 'Ġ3', 'Ġ:', 'ĠUn', 'recorded', 'ĠChronicles', 'Ġ(', 'ĠJapanese', 'Ġ:', 'Ġæ', 'Ī', '¦', 'å', 'ł', '´', 'ãģ®', 'ãĥ´ãĤ¡', 'ãĥ«', 'ãĤŃ', 'ãĥ¥

## 9. GPT-Style Transformer with GPT-2 Tokenizer

This model also uses random weights.

Only the tokenizer is pretrained.

The model itself must still learn from WikiText-2.


In [17]:
def get_batch_gpt2(split):
    # Select training or validation token data.
    source = train_gpt2_data if split == "train" else val_gpt2_data

    # Choose random start positions.
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))

    # Create input token sequences.
    x = torch.stack([source[i:i + block_size] for i in ix])

    # Move tensors to CPU or GPU.
    return x.to(device)

# Inspect one GPT-2 token batch.
xb = get_batch_gpt2("train")
print("x shape:", xb.shape)
print("First 20 token IDs:", xb[0][:20].tolist())
print("First 20 tokens:", tokenizer.convert_ids_to_tokens(xb[0][:20].tolist()))


x shape: torch.Size([32, 256])
First 20 token IDs: [25564, 2488, 12, 31, 36784, 286, 19101, 220, 628, 360, 3674, 3713, 1471, 82, 5362, 288, 705, 3163, 1840, 64]
First 20 tokens: ['ĠCardinal', 'Ġ@', '-', '@', 'Ġprotector', 'Ġof', 'ĠPortugal', 'Ġ', 'ĊĊ', 'ĠD', 'omen', 'ico', 'ĠOr', 's', 'ini', 'Ġd', "Ġ'", 'Ar', 'agon', 'a']


In [18]:
# Define a small GPT-2 style configuration for GPT-2 tokenizer tokens.
gpt2_config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=128,
    n_layer=4,
    n_head=4,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

# Create an untrained GPT-2 style model with random weights.
gpt2_random = GPT2LMHeadModel(gpt2_config).to(device)

print("GPT-2 style model parameters:", gpt2_random.num_parameters())


GPT-2 style model parameters: 7259008


In [19]:
# Create the optimizer.
gpt2_optimizer = torch.optim.AdamW(gpt2_random.parameters(), lr=1e-3)

# Define number of training steps.
gpt2_steps = 3000

# Train the untrained GPT-2 style model.
gpt2_random.train()

for step in range(gpt2_steps):
    # Get one GPT-2 token batch.
    xb = get_batch_gpt2("train")

    # Pass labels=xb because Hugging Face shifts labels internally.
    outputs = gpt2_random(input_ids=xb, labels=xb)

    # Extract the cross-entropy loss.
    loss = outputs.loss

    # Reset gradients.
    gpt2_optimizer.zero_grad()

    # Compute gradients.
    loss.backward()

    # Update model parameters.
    gpt2_optimizer.step()

    # Print progress.
    if step % eval_interval == 0:
        print(f"Step {step}, loss: {loss.item():.4f}")


Step 0, loss: 10.8534
Step 300, loss: 6.0740
Step 600, loss: 5.4577
Step 900, loss: 5.1297
Step 1200, loss: 5.0784
Step 1500, loss: 4.8900
Step 1800, loss: 4.6043
Step 2100, loss: 4.3631
Step 2400, loss: 4.3948
Step 2700, loss: 4.3900


In [21]:
# Define a starting prompt.
prompt = "The capital of France is"

# Convert prompt into GPT-2 token IDs.
input_ids = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=False).to(device)

# Generate new GPT-2 tokens.
output_ids = gpt2_random.generate(
    input_ids,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    pad_token_id=tokenizer.eos_token_id,
)

# Decode generated token IDs into readable text.
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


The capital of France is a popular tourist attraction . 

 The city is a permanent library of two floors in the United Kingdom of India . The museum is a four floors of
